In [1]:
import getml
import mlflow
import getml.mlflow

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")

if not mlflow.search_experiments(filter_string="name='interstate94'"):
    mlflow.create_experiment("interstate94")
mlflow.set_experiment("interstate94")

getml.mlflow.autolog()

In [3]:
getml.engine.launch()

Launching ./getML --allow-push-notifications=true --allow-remote-ips=false --home-directory=/home/manuel/.getML --in-memory=true --install=false --launch-browser=true --log=false --project-directory=/home/manuel/.getML/projects in /home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/getml/.getML/getml-community-1.5.0-amd64-linux...
Launched the getML Engine. The log output will be stored in /home/manuel/.getML/logs/getml_20241213165540.log


In [4]:
getml.engine.set_project("interstate94")

Output()

Connected to project 'interstate94'.

In [5]:
traffic = getml.datasets.load_interstate94(roles=False, units=False)

In [6]:
traffic.set_role("ds", getml.data.roles.time_stamp)
traffic.set_role("holiday", getml.data.roles.categorical)
traffic.set_role("traffic_volume", getml.data.roles.target)

In [7]:
split = getml.data.split.time(traffic, "ds", test=getml.data.time.datetime(2018, 3, 15))

In [8]:
time_series = getml.data.TimeSeries(
    population=traffic,
    split=split,
    time_stamps="ds",
    horizon=getml.data.time.hours(1),
    memory=getml.data.time.days(7),
    lagged_targets=True,
)

pipe = getml.pipeline.Pipeline(
    tags=["memory: 7d", "horizon: 1h", "fast_prop"],
    data_model=time_series.data_model,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[
        getml.feature_learning.FastProp(
            loss_function=getml.feature_learning.loss_functions.SquareLoss,
            num_threads=1,
            num_features=20,
        )
    ],
    predictors=[getml.predictors.XGBoostRegressor()],
)

In [9]:
pipe.fit(time_series.train)

2024/12/13 16:55:42 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b5472bfa1fb94b3997f2680e4d1267a2', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current getml workflow
2024/12/13 16:55:42 INFO mlflow.tracking._tracking_service.client: 🏃 View run nosy-lark-634 at: http://localhost:5000/#/experiments/844494434965818253/runs/b5472bfa1fb94b3997f2680e4d1267a2.
2024/12/13 16:55:42 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Checking data model...

Output()

OK.

Output()

Trained pipeline.

2024/12/13 16:55:50 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during getml autologging: HTTPConnectionPool(host='localhost', port=1709): Max retries exceeded with url: /getcpuusage/ (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ff5101fbdd0>: Failed to establish a new connection: [Errno 111] Connection refused'))


Time taken: 0:00:08.420627.



Pipeline(data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop', 'container-50uUTK'])

In [10]:
pipe.score(time_series.test)

2024/12/13 16:55:50 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '53cea05983c84aa99ed1b19a76a0a720', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current getml workflow


Output()

2024/12/13 16:55:52 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2024/12/13 16:55:52 WARNING mlflow.models.evaluation.default_evaluator: Skip logging model explainability insights because the shap explainer None requires all feature values to be numeric, and each feature column must only contain scalar values.


Output()

2024/12/13 16:55:52 INFO mlflow.tracking._tracking_service.client: 🏃 View run funny-mouse-859 at: http://localhost:5000/#/experiments/844494434965818253/runs/53cea05983c84aa99ed1b19a76a0a720.
2024/12/13 16:55:52 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


,date time,set used,target,mae,rmse,rsquared
0,2024-12-13 16:55:50,train,traffic_volume,200.4302,299.2045,0.9768
1,2024-12-13 16:55:52,test,traffic_volume,179.9515,269.631,0.9816
